# 문제1

In [ ]:
import cv2

img = cv2.imread('./images/night_ops/IMG001.png')

In [ ]:
import matplotlib.pyplot as plt

img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

plt.imshow(img_rgb)
plt.title('Loaded Image')
plt.axis('off')
plt.show()

In [ ]:
resized = cv2.resize(img, (256, 256))

In [ ]:
resized.shape

In [ ]:
gray = cv2.cvtColor(resized, cv2.COLOR_BGR2GRAY)

In [ ]:
plt.imshow(gray)
plt.title('Gray Image')
plt.axis('off')
plt.show()

In [ ]:
import numpy as np

brightness = np.mean(gray)

In [ ]:
gray

In [ ]:
brightness

In [ ]:
image_dir = "./images/night_ops/"
image_files = ["IMG001.png", "IMG002.png", "IMG003.png"]

In [ ]:
results = []

In [ ]:
import os

for file_name in image_files:
    image_path = os.path.join(image_dir, file_name)
    img = cv2.imread(image_path)
    resized = cv2.resize(img, (256, 256))
    gray = cv2.cvtColor(resized, cv2.COLOR_BGR2GRAY)
    brightness = np.mean(gray)
    results.append({"filename": file_name, "brightness": brightness})

In [ ]:
results

In [ ]:
import pandas as pd

df = pd.DataFrame(results)
df.to_csv("brightness_result.csv", index=False)

In [ ]:
brightest = df.loc[df["brightness"].idxmax()]

In [ ]:
print("가장 밝은 이미지 파일명:", brightest["filename"])

### 정리

In [ ]:
import cv2
import os
import numpy as np
import pandas as pd

# 이미지 폴더 경로 설정
image_dir = "./images/night_ops/"
image_files = ["IMG001.png", "IMG002.png", "IMG003.png"]

# 밝기 결과 저장 리스트 생성
results = []

# 1. 256x256 리사이즈
# 2. Grayscale 변환 및 밝기 평균 계산
for file_name in image_files:
    image_path = os.path.join(image_dir, file_name)
    img = cv2.imread(image_path)  # BGR 이미지 불러오기
    if img is None:
        print(f"{file_name} 로드 실패")
        continue
    resized = cv2.resize(img, (256, 256))             # 리사이즈 (1)
    gray = cv2.cvtColor(resized, cv2.COLOR_BGR2GRAY)  # 그레이스케일 변환 (2)
    brightness = np.mean(gray)                        # 평균 밝기 계산 (2)
    results.append({"filename": file_name, "brightness": brightness})

# 3. CSV로 저장
df = pd.DataFrame(results)
df.to_csv("brightness_result.csv", index=False)

# 4. 가장 밝은 이미지 파일명 출력
brightest = df.loc[df["brightness"].idxmax()]
print("가장 밝은 이미지 파일명:", brightest["filename"])

---

# 문제2

In [ ]:
import cv2

img = cv2.imread('./images/blueprints/BP003.png')

In [ ]:
import matplotlib.pyplot as plt

img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(6, 6))
plt.imshow(img_rgb)
plt.title("Loaded Image")
plt.axis('off')
plt.show()

In [ ]:
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
blurred = cv2.GaussianBlur(gray, (5, 5), 0)
edges = cv2.Canny(blurred, 50, 150)

In [ ]:
plt.figure(figsize=(15, 4))

plt.subplot(1, 3, 1)
plt.imshow(gray, cmap='gray')
plt.title('Grayscale')
plt.axis('off')

plt.subplot(1, 3, 2)
plt.imshow(blurred, cmap='gray')
plt.title('Blurred')
plt.axis('off')

plt.subplot(1, 3, 3)
plt.imshow(edges, cmap='gray')
plt.title('Canny Edges')
plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

In [ ]:
contours

In [ ]:
img_with_rects = img.copy()

In [ ]:
for cnt in contours:
    approx = cv2.approxPolyDP(cnt, 0.02 * cv2.arcLength(cnt, True), True)
    if len(approx) == 4 and cv2.isContourConvex(approx):
        x, y, w, h = cv2.boundingRect(approx)
        print(x, y, w, h)

        cv2.rectangle(img_with_rects, (x, y), (x + w, y + h), (0, 0, 255), 2)

In [ ]:
img_rgb = cv2.cvtColor(img_with_rects, cv2.COLOR_BGR2RGB)

# 결과 이미지 출력
plt.figure(figsize=(8, 8))
plt.imshow(img_rgb)
plt.title('Detected Rectangles')
plt.axis('off')
plt.show()

In [ ]:
image_dir = "./images/blueprints/"
image_files = ["BP001.png", "BP002.png", "BP003.png"]

In [ ]:
rectangle_data = []

In [ ]:
import os

for image_name in image_files:

    image_path = os.path.join(image_dir, image_name)
    img = cv2.imread(image_path)
    if img is None:
        print(f"이미지 로드 실패: {image_path}")
        continue

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    edges = cv2.Canny(blurred, 50, 150)

    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    rectangle_count = 0

    for cnt in contours:
        approx = cv2.approxPolyDP(cnt, 0.02 * cv2.arcLength(cnt, True), True)
        if len(approx) == 4 and cv2.isContourConvex(approx):
            x, y, w, h = cv2.boundingRect(approx)
            rectangle_count += 1
            rectangle_data.append({
                "filename": image_name,
                "x": x,
                "y": y,
                "width": w,
                "height": h,
                "rectangle_count": rectangle_count
            })

    if rectangle_count == 0:
        rectangle_data.append({
            "filename": image_name,
            "x": None,
            "y": None,
            "width": None,
            "height": None,
            "rectangle_count": rectangle_count
        })

In [ ]:
import pandas as pd

df = pd.DataFrame(rectangle_data)
df.to_csv("rectangles.csv", index=False)

### 정리

In [ ]:
import cv2
import numpy as np
import pandas as pd
import os

# 이미지 경로 리스트
image_dir = "./images/blueprints/"
image_files = ["BP001.png", "BP002.png", "BP003.png"]

rectangle_data = []

for image_name in image_files:
    # 이미지 로드
    image_path = os.path.join(image_dir, image_name)
    img = cv2.imread(image_path)
    if img is None:
        print(f"이미지 로드 실패: {image_path}")
        continue

    # 그레이스케일 + 전처리
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    edges = cv2.Canny(blurred, 50, 150)

    # 1. 윤곽선 검출
    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    rectangle_count = 0

    # 2. 사각형 필터링
    for cnt in contours:
        approx = cv2.approxPolyDP(cnt, 0.02 * cv2.arcLength(cnt, True), True)
        if len(approx) == 4 and cv2.isContourConvex(approx):
            x, y, w, h = cv2.boundingRect(approx)
            rectangle_count += 1
            rectangle_data.append({
                "filename": image_name,
                "x": x,
                "y": y,
                "width": w,
                "height": h,
                "rectangle_count": rectangle_count
            })

    # 3. 이미지 당 사각형 수 카운트 기록
    if rectangle_count == 0:
        rectangle_data.append({
            "filename": image_name,
            "x": None,
            "y": None,
            "width": None,
            "height": None,
            "rectangle_count": rectangle_count
        })

# 3~4. CSV 저장
df = pd.DataFrame(rectangle_data)
df.to_csv("rectangles.csv", index=False)

---

# 문제3

In [ ]:
import torch

model = torch.hub.load('ultralytics/yolov5', 'yolov5s', pretrained=True)

In [ ]:
from PIL import Image

img = Image.open('./images/surv_imgs/CAM001.png')

In [ ]:
results = model(img)

In [ ]:
results

In [ ]:
results.render()

In [ ]:
img_with_boxes = results.ims[0]

In [ ]:
img_with_boxes

In [ ]:
import cv2

img_rgb = cv2.cvtColor(img_with_boxes, cv2.COLOR_BGR2RGB)

In [ ]:
%matplotlib inline

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 10))
plt.imshow(img_rgb)
plt.title("YOLOv5 Detection")
plt.axis('off')
plt.show()

In [ ]:
detections = []

In [ ]:
person_count = 0
vehicle_count = 0

In [ ]:
for *box, conf, cls in results.xyxy[0].tolist():
    x1, y1, x2, y2 = map(int, box)
    class_id = int(cls)
    class_name = results.names[class_id]

    if class_name in ['person']:
        person_count += 1
    elif class_name in ['car', 'truck', 'bus']:
        vehicle_count += 1
    else:
        continue

    detections.append({
        "class": class_name,
        "x1": x1,
        "y1": y1,
        "x2": x2,
        "y2": y2
    })

In [ ]:
detections

In [ ]:
import pandas as pd

df = pd.DataFrame(detections)
df.to_csv("detections.csv", index=False)

In [ ]:
print(f"탐지된 사람 수: {person_count}")
print(f"탐지된 차량 수: {vehicle_count}")

### 정리

In [ ]:
import torch
import pandas as pd
import os
from PIL import Image

# YOLOv5 사전학습 모델 로드 (COCO 클래스 학습됨)
model = torch.hub.load('ultralytics/yolov5', 'yolov5s', pretrained=True)

# 이미지 경로 설정
image_dir = './images/surv_imgs/'
image_files = ['CAM001.png', 'CAM002.png', 'CAM003.png']

# 결과 저장 리스트 생성
detections = []

# 사람/차량 카운터 선언
person_count = 0
vehicle_count = 0

for file_name in image_files:
    img_path = os.path.join(image_dir, file_name)
    img = Image.open(img_path)

    # 1. 객체 탐지
    # 2. 클래스명, 좌표 출력
    results = model(img)
    for *box, conf, cls in results.xyxy[0].tolist():
        x1, y1, x2, y2 = map(int, box)
        class_id = int(cls)
        class_name = results.names[class_id]

        # 클래스명이 person 또는 vehicle 관련인지 확인
        if class_name in ['person']:
            person_count += 1
        elif class_name in ['car', 'truck', 'bus']:
            vehicle_count += 1
        else:
            continue  # 그 외 클래스는 무시

        detections.append({
            "filename": file_name,
            "class": class_name,
            "x1": x1,
            "y1": y1,
            "x2": x2,
            "y2": y2
        })

# 3. CSV 저장
df = pd.DataFrame(detections)
df.to_csv("detections.csv", index=False)

# 4. 합산 출력
print(f"탐지된 사람 수: {person_count}")
print(f"탐지된 차량 수: {vehicle_count}")